# WANI PVA Pipeline in Colab

## Steps to execute the pipeline
1. Change the run time to T4-GPU
    - Select `Runtime` on the top-left horizontal taskbar (the one that has `File-Edit-View-Insert...`)
    - Select `Change runtime type`
    - Select `T4 GPU`

2. Modify these two things in the next cell: `SCENARIOS_CHOICE` and `PATH_TO_YOUR_SPEGG_IN_DRIVE` to select the population simulation scenario and the location in your google drive where you want to save the simulation results to

3. Click `Run all` to run all the cells

> **__NOTE:__**
>
> Immediately after hitting `Run all`, the notebook will ask you for **your permission for it to access your Google Drive account**. Please check all the checkboxes to continue the execution!
>
> While running, **always leave your colab notebook active!!**, otherwise as soon as the running notebook goes to sleep, its runtime will get disconnected and all the memory will get flushed, which means you have to re-run the notebook from the start!



## Execution Start

### Users' level variables

In [ ]:
# DO NOT MODIFY THIS LINE
SCENARIOS = ['fishScenarios.R', 'invertebrateScenarios.R', 'CorrelatedfishScenarios.R']
#### TODO: CHOOSE ONE OF THOSE SCENARIOS AND ENTER ITS INDEX HERE
SCENARIOS_CHOICE = 0 ## CHANGE THIS

#### TODO: ENTER THE PATH TO THE FOLDER IN YOUR GG DRIVE TO WHICH YOU WANT TO SAVE THIS SIMULATION RESULTS
PATH_TO_YOUR_SPEGG_IN_DRIVE = 'Colab Notebooks' ## CHANGE THIS


# TESTING USERS' INPUTS -- DO NOT MODIFY THIS LINE
assert SCENARIOS_CHOICE in range(len(SCENARIOS))

### Mount Drive

In [ ]:
import os, shutil, google.colab

In [ ]:
google.colab.drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Clone sPEGG

In [ ]:
# If you have not already cloned spegg, please run the following code cell:
!git clone --recursive https://github.com/kewok/pspm_pva.git

Cloning into 'pspm_pva'...
remote: Enumerating objects: 689, done.
remote: Counting objects: 100% (689/689), done.
remote: Compressing objects: 100% (437/437), done.
remote: Total 689 (delta 229), reused 684 (delta 224), pack-reused 0 (from 0)
Receiving objects: 100% (689/689), 3.15 MiB | 7.77 MiB/s, done.
Resolving deltas: 100% (229/229), done.
Submodule 'header/util/rapidcsv' (https://github.com/d99kris/rapidcsv.git) registered for path 'header/util/rapidcsv'
Cloning into '/content/pspm_pva/header/util/rapidcsv'...
remote: Enumerating objects: 2695, done.        
remote: Counting objects: 100% (693/693), done.        
remote: Compressing objects: 100% (194/194), done.        
remote: Total 2695 (delta 571), reused 506 (delta 495), pack-reused 2002 (from 2)        
Receiving objects: 100% (2695/2695), 15.61 MiB | 16.85 MiB/s, done.
Resolving deltas: 100% (1822/1822), done.
Submodule path 'header/util/rapidcsv': checked out 'c58633ddaf19d305fce0bf59eee9e6222d071797'


### Move into spegg directory

In [ ]:
%cd pspm_pva

/content/pspm_pva


### Install libconfig package

In [ ]:
!sudo apt-get update
!sudo apt-get install libconfig++-dev

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,473 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,914 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu j

### Compile the codebase

In [ ]:
!make

mkdir objdir
nvcc -c -Xcompiler -O3 -arch=sm_75 -I./header  ./src/cuda/species/inds.cu -o objdir/inds.o
nvcc -c -Xcompiler -O3 -arch=sm_75 -I./header  ./src/cuda/species/inds_stochastic.cu -o objdir/inds_stochastic.o
nvcc -c -Xcompiler -O3 -arch=sm_75 -I./header  ./src/cuda/species/inds_stochastic_migratory.cu -o objdir/inds_stochastic_migratory.o
./src/cuda/species/inds_stochastic_migratory.cu(31): warning #1650-D: result of call is not used
    fscanf(Migration_Probabilities_File, "%lf\n", &test);
    ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

./src/cuda/species/inds_stochastic_migratory.cu(31): warning #1650-D: result of call is not used
    fscanf(Migration_Probabilities_File, "%lf\n", &test);
    ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

./src/cuda/species/inds_stochastic_migratory.cu: In constructor ‘inds_stochastic_migratory::inds_stochastic_migratory(int, int, int, int, int)’:
./src/cuda/species/inds

### Compile the project and run its simulation

#### Move to project directory

In [ ]:
%cd Examples/WaniPVA/

/content/pspm_pva/Examples/WaniPVA


#### Compile

In [ ]:
!make

mkdir objdir
nvcc -c -Xcompiler -O3 -arch=sm_75 -I./../../header/ -I./header  main.cu -o objdir/main.o
nvcc -c -Xcompiler -O3 -arch=sm_75 -I./../../header/ -I./header  ./src/genotype_phenotype_map.cu -o objdir/genotype_phenotype_map.o
./src/genotype_phenotype_map.cu(34): warning #940-D: missing return statement at end of non-void function "GenotypePhenotypeMap::create_genotype_phenotype_map"
   }
   ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

./src/genotype_phenotype_map.cu(34): warning #940-D: missing return statement at end of non-void function "GenotypePhenotypeMap::create_genotype_phenotype_map"
   }
   ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

./src/genotype_phenotype_map.cu: In static member function ‘static GenotypePhenotypeMap* GenotypePhenotypeMap::create_genotype_phenotype_map(inds*, int, int, int)’:
./src/genotype_phenotype_map.cu:34:8: warning: control reaches end of non-void function []8;;https:

#### Generate configuration files based on user-specified scenario and replicate the simulation 100 times

In [ ]:
!cd configFileGenerator/ && Rscript {SCENARIOS[SCENARIOS_CHOICE]} && mv *_config.txt ..

In [ ]:
!Rscript Repeater.R

Total Time elapsed: 915s, 998ms
Total Time elapsed: 919s, 869ms
Total Time elapsed: 924s, 54ms
Total Time elapsed: 922s, 796ms
^C


### Move results to Drive

In [ ]:
PATH_TO_YOUR_SPEGG_IN_DRIVE += "/pspm_pva"
full_path_destination = os.path.join("/content/drive/My Drive/", PATH_TO_YOUR_SPEGG_IN_DRIVE)
source_path = "/content/pspm_pva/Examples/WaniPVA/Results"
shutil.copytree(source_path, full_path_destination)

'/content/drive/My Drive/Colab Notebooks/pspm_pva'

8. For future uses, mount your Drive back on your Colab notebook by running these lines in a Colab cell:
```
from google.colab import drive
drive.mount("/content/drive")
```
Also, you may wish to redownload `libconfig` (shouldn't take more than 20 seconds)
```
!sudo apt-get update
!sudo apt-get install libconfig++-dev
```
Then you can manually navigate into `MyDrive` directory until you find spegg directory.